In [1]:
!pip install qiskit-ibm-runtime


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 35.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 58.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.5/541.5 kB 21.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 56.7 MB/s  0:00:00
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.9.0.post0
    Uninstalling python-dateutil-2.9.0.post0:
      Successfully uninstalled python-dateutil-2.9.0.post0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [qiskit-ibm-runtime]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
holidays 0.107 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 3.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 3.0.0 which is incompatible.
google-cloud-bigque

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Save your credentials (run this once)
QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="put_your_ibm_token_here",
    instance="put_your_ibm_instance_id_here",
    overwrite=True
)

# Now you can connect and find a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
print(f"Connected successfully to: {backend.name}")

Connected successfully to: ibm_kingston



In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import SamplerV2 as Sampler

print("Building the 12-qubit circuit...")
# FIX: Added 6 classical bits (12 quantum, 6 classical)
qc = QuantumCircuit(12, 6)

# 1. Setup complex data on 6 core qubits (Qubits 0-5)
for i in range(6):
    qc.h(i)
    qc.ry(0.5, i)

# 2. Setup redundancy (Entangle core with ancillas 6-11)
for i in range(6):
    qc.cx(i, 6 + i)

# 3. COMPRESS (Uncompute the redundancy)
for i in range(6):
    qc.cx(i, 6 + i)

# 4. Measure ONLY the 6 ancilla qubits (Qubits 6-11) into classical bits (0-5)
qc.measure([6, 7, 8, 9, 10, 11], [0, 1, 2, 3, 4, 5])

print(f"Transpiling for {backend.name}...")
qc_transpiled = transpile(qc, backend=backend, optimization_level=1)

print("Sending job to real quantum hardware...")
sampler = Sampler(backend)
job = sampler.run([qc_transpiled], shots=1000)
print(f"Job ID: {job.job_id()}")
print("Waiting for results (this takes about 10-30 seconds)...")

result = job.result()
counts = result[0].data.c.get_counts()

# Calculate how many times the ancillas perfectly collapsed to |000000>
success_count = counts.get('000000', 0)
success_rate = (success_count / 1000) * 100

print("\n--- REAL HARDWARE RESULTS ---")
print(f"Ancillas reset to |000000> successfully: {success_count} out of 1000 shots")
print(f"Hardware Compression Success Rate: {success_rate:.2f}%")
print("Top 3 measurement outcomes:", list(counts.keys())[:3])

Building the 12-qubit circuit...
Transpiling for ibm_kingston...
Sending job to real quantum hardware...
/tmp/ipykernel_9728/2886357751.py:28: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = Sampler(backend)
Job ID: dbgc22djm0ks73egaq2g
Waiting for results (this takes about 10-30 seconds)...

--- REAL HARDWARE RESULTS ---
Ancillas reset to |000000> successfully: 956 out of 1000 shots
Hardware Compression Success Rate: 95.60%
Top 3 measurement outcomes: ['000000', '000100', '010000']


In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import SamplerV2 as Sampler
import time

start_time = time.time()

print("Building the 20-qubit circuit (10 core, 10 ancilla)...")
# 20 Quantum bits, 10 Classical bits for the ancillas
qc20 = QuantumCircuit(20, 10)

# 1. Setup complex data on 10 core qubits (0-9)
for i in range(10):
    qc20.h(i)
    qc20.ry(0.5, i)

# 2. Setup redundancy (Entangle core 0-9 with ancillas 10-19)
for i in range(10):
    qc20.cx(i, 10 + i)

# 3. COMPRESS (Uncompute the redundancy)
for i in range(10):
    qc20.cx(i, 10 + i)

# 4. Measure ONLY the 10 ancilla qubits (10-19) into classical bits (0-9)
qc20.measure([10, 11, 12, 13, 14, 15, 16, 17, 18, 19], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

print(f"Transpiling for {backend.name}...")
qc20_transpiled = transpile(qc20, backend=backend, optimization_level=1)

print("Sending 20-qubit job to real quantum hardware...")
sampler = Sampler(backend)
job20 = sampler.run([qc20_transpiled], shots=1000)
print(f"20-Qubit Job ID: {job20.job_id()}")
print("Waiting for results...")

result20 = job20.result()
counts20 = result20[0].data.c.get_counts()

success_count20 = counts20.get('0000000000', 0)
success_rate20 = (success_count20 / 1000) * 100

print("\n--- 20-QUBIT REAL HARDWARE RESULTS ---")
print(f"Ancillas reset to |0000000000> successfully: {success_count20} out of 1000 shots")
print(f"Hardware Compression Success Rate: {success_rate20:.2f}%")
print("Top 3 measurement outcomes:", list(counts20.keys())[:3])
print(f"Total execution time: {time.time() - start_time:.1f} seconds")

Building the 20-qubit circuit (10 core, 10 ancilla)...
Transpiling for ibm_kingston...
Sending 20-qubit job to real quantum hardware...
/tmp/ipykernel_9728/2770205110.py:31: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = Sampler(backend)
20-Qubit Job ID: dbgc310iudkc73e46hhg
Waiting for results...

--- 20-QUBIT REAL HARDWARE RESULTS ---
Ancillas reset to |0000000000> successfully: 901 out of 1000 shots
Hardware Compression Success Rate: 90.10%
Top 3 measurement outcomes: ['0000000000', '0001000000', '0000000100']
Total execution time: 29.1 seconds


In [ ]:
import os
import hashlib
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import SamplerV2 as Sampler
import time

print("1. Creating a REAL 10 MB text file with random data...")
file_path = "10MB_file.txt"
with open(file_path, "wb") as f:
    f.write(os.urandom(10 * 1024 * 1024)) # Generates 10 MB of random data

file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
print(f"File '{file_path}' created. Size: {file_size_mb:.2f} MB")

print("\n2. Reading data from the 10MB file to encode into the quantum circuit...")
with open(file_path, "rb") as f:
    file_bytes = f.read(10) # Read first 10 bytes to seed the 20-qubit circuit

# Convert bytes to rotation angles for the quantum gates
angles = [(b / 255.0) * 3.14159 for b in file_bytes]
print(f"Extracted bytes: {list(file_bytes)}")
print(f"Mapped to quantum angles: {[round(a, 2) for a in angles]}")

print("\n3. Building the 20-qubit circuit using the 10MB file's data...")
# We use 20 qubits (10 core, 10 ancilla)
qc = QuantumCircuit(20, 10)

# Prepare core data using the 10MB file's bytes
for i in range(10):
    qc.h(i)
    qc.ry(angles[i], i) # Using data from the file!

# Setup redundancy (entangle with ancillas)
for i in range(10):
    qc.cx(i, 10 + i)

# COMPRESS
for i in range(10):
    qc.cx(i, 10 + i)

# Measure ancillas
qc.measure([10, 11, 12, 13, 14, 15, 16, 17, 18, 19], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

print(f"\n4. Transpiling for {backend.name}...")
qc_transpiled = transpile(qc, backend=backend, optimization_level=1)

print("5. Sending job to real IBM Quantum Hardware...")
sampler = Sampler(backend)
job = sampler.run([qc_transpiled], shots=1000)
print(f"Job ID: {job.job_id()}")
print("Waiting for results...")

result = job.result()
counts = result[0].data.c.get_counts()

success_count = counts.get('0000000000', 0)
success_rate = (success_count / 1000) * 100

print("\n" + "="*40)
print("--- REAL HARDWARE 10MB FILE COMPRESSION ---")
print("="*40)
print(f"File Size Simulated: {file_size_mb:.2f} MB")
print(f"Ancillas reset to |0> successfully: {success_count} out of 1000")
print(f"Hardware Compression Success Rate: {success_rate:.2f}%")
print("Top 3 outcomes:", list(counts.keys())[:3])

1. Creating a REAL 10 MB text file with random data...
File '10MB_file.txt' created. Size: 10.00 MB

2. Reading data from the 10MB file to encode into the quantum circuit...
Extracted bytes: [146, 127, 233, 60, 32, 79, 37, 225, 139, 202]
Mapped to quantum angles: [1.8, 1.56, 2.87, 0.74, 0.39, 0.97, 0.46, 2.77, 1.71, 2.49]

3. Building the 20-qubit circuit using the 10MB file's data...

4. Transpiling for ibm_kingston...
5. Sending job to real IBM Quantum Hardware...
/tmp/ipykernel_9728/4254352331.py:48: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = Sampler(backend)
Job ID: dbgc390iudkc73e46i70
Waiting for results...

========================================
--- REAL HARDWARE 10MB FILE COMPRESSION ---
========================================
File Size Simulated: 10.00 MB
Ancillas reset to |0> successfully: 903 out of 1000
Hardware Compression Success Rate: 90.30%
Top 3 outcomes: ['0000000000', '0100000000', '0000000100']


In [ ]:
import os
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import SamplerV2 as Sampler

print("1. Creating real 10 MB file...")
with open("10MB_file.txt", "wb") as f:
    f.write(os.urandom(10 * 1024 * 1024))

with open("10MB_file.txt", "rb") as f:
    file_bytes = f.read(10)
angles = [(b / 255.0) * 3.14159 for b in file_bytes]

print("2. Building Full Cycle (Setup -> Compress -> Decompress)...")
setup = QuantumCircuit(20)
for i in range(10):
    setup.h(i)
    setup.ry(angles[i], i)
    setup.cx(i, 10 + i) # Entangle redundancy

comp = QuantumCircuit(20)
decomp = QuantumCircuit(20)
for i in range(10):
    comp.cx(i, 10 + i)
    decomp.cx(i, 10 + i) # Decompression

# Full protocol
protocol = setup.compose(comp).compose(decomp)

print("3. Applying Compute-Uncompute to verify data restoration...")
# If decompression works, protocol equals setup.
# So, protocol + setup.inverse() should yield all |000...0>
full_test = protocol.compose(setup.inverse())
full_test.measure_all()

print(f"4. Transpiling for {backend.name}...")
qc_transpiled = transpile(full_test, backend=backend, optimization_level=1)

print("5. Sending to hardware...")
sampler = Sampler(backend)
job = sampler.run([qc_transpiled], shots=1000)
print(f"Job ID: {job.job_id()}")
print("Waiting for results...")

result = job.result()
counts = result[0].data.meas.get_counts()

# Check how many times we got all 20 qubits back as 0
success_count = counts.get('0'*20, 0)
success_rate = (success_count / 1000) * 100

print("\n" + "="*40)
print("--- FULL CYCLE (COMPRESS + DECOMPRESS) ---")
print("="*40)
print(f"Data completely restored (20 zeros): {success_count} out of 1000")
print(f"Restoration Success Rate: {success_rate:.2f}%")
print("Top 3 outcomes:", list(counts.keys())[:3])

1. Creating real 10 MB file...
2. Building Full Cycle (Setup -> Compress -> Decompress)...
3. Applying Compute-Uncompute to verify data restoration...
4. Transpiling for ibm_kingston...
5. Sending to hardware...
/tmp/ipykernel_9728/405142563.py:39: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = Sampler(backend)
Job ID: dbgc4i1j4t8s73ac2bcg
Waiting for results...

========================================
--- FULL CYCLE (COMPRESS + DECOMPRESS) ---
========================================
Data completely restored (20 zeros): 214 out of 1000
Restoration Success Rate: 21.40%
Top 3 outcomes: ['00000000000000000001', '00000000000000000000', '00000100000000000001']
